# Tarea 4. Modelo de Lenguje de Política

Procesamiento de Lenguaje Natural 

Eric Lemus Avalos 

## Textos de las Mañaneras 

In [165]:
import os 
import re
import time
import random
import numpy as np 
import polars as pl
import pandas as pd

In [161]:
data = pl.read_parquet(str(os.path.dirname(os.getcwd())) + str('\\data\\transcripts.parquet'))

In [16]:
data_pandas = data.to_pandas()
data_pandas

,Conferencia,Contenido,Autor,Fecha,Repeticion
0,Estenografica mensaje de la presidenta de los ...,"PRESIDENTA CLAUDIA SHEINBAUM PARDO:Amigas, ami...",Sheinbaum,2024-10-01,1
1,Estenografica toma de protesta ante el pueblo ...,MODERADORA 1: Mexicanas y mexicanos:Estamos en...,Sheinbaum,2024-10-01,2
2,Estenografica toma de protesta de claudia shei...,"IFIGENIA MARTÍNEZ Y HERNÁNDEZ, PRESIDENTA DE L...",Sheinbaum,2024-10-01,3
3,Estenografica conferencia de prensa la preside...,PRESIDENTA CLAUDIA SHEINBAUM PARDO:Buenos días...,Sheinbaum,2024-10-02,1
4,Estenografica conferencia de prensa de la pres...,PRESIDENTA CLAUDIA SHEINBAUM PARDO:Buenos días...,Sheinbaum,2024-10-03,1
...,...,...,...,...,...
2099,Version estenografica de la conferencia de pre...,"2024: Año de Felipe Carrillo Puerto, benemérit...",AMLO,2024-09-27,1
2100,Version estenografica entrega de acueducto y d...,"2024: Año de Felipe Carrillo Puerto, benemérit...",AMLO,2024-09-27,2
2101,Version estenografica inauguracion zona de rie...,"2024: Año de Felipe de Carrillo Puerto, benemé...",AMLO,2024-09-28,1
2102,Version estenografica tren maya y entrega de r...,"2024: Año de Felipe Carrillo Puerto, benemérit...",AMLO,2024-09-29,1


In [17]:
data_pandas = data.to_pandas()
transcripciones =  data_pandas['Contenido'].to_list()

In [18]:
def eliminar_palabras_largas(texto):
    palabras = texto.split()
    palabras_filtradas = [palabra for palabra in palabras if len(palabra) <= 30]
    return " ".join(palabras_filtradas)

transcripciones_limpias = [eliminar_palabras_largas(transcripcion) for transcripcion in transcripciones]

In [22]:
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt')

oraciones = []
for transcripcion in transcripciones_limpias:
    oraciones_transcripcion = sent_tokenize(transcripcion, language='spanish')
    oraciones.extend(oraciones_transcripcion)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ericl\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [23]:
with open('corpus_oraciones.txt', 'w', encoding='utf-8') as archivo:
    for oracion in oraciones:
        archivo.write(oracion + '\n')  

In [24]:
print(f"Numero de oraciones: {len(oraciones)}")

Numero de oraciones: 306760


In [25]:
from nltk.tokenize.punkt import PunktSentenceTokenizer

tokenizer = PunktSentenceTokenizer()

oraciones = []
for transcripcion in transcripciones_limpias:
    oraciones_transcripcion = tokenizer.tokenize(transcripcion)
    oraciones.extend(oraciones_transcripcion)

with open('corpus_PunkSentenceTk.txt', 'w', encoding='utf-8') as archivo:
    for oracion in oraciones:
        archivo.write(oracion + '\n')

In [26]:
print(f"Numero de oraciones: {len(oraciones)}")

Numero de oraciones: 306876


## Modelo de Lenguaje y Evaluación


### 1. Preproceso de los textos 

Para el preprocesado vamos a pasar el texto a minúsculas y quitar algunos urls así como algunos caracteres de puntuación. Agregaremos los tokens de inicio y fin de oración ($<s>$ , $</s>$). 


In [27]:
# Leemos las oraciones creadas con PunktSentenceTokenizer 
with open('corpus_PunkSentenceTk.txt', 'r', encoding='utf-8') as archivo:
    oraciones = archivo.readlines()

In [28]:
url_pattern = r'https?://\S+|www\.\S+'

oraciones = [re.sub(url_pattern, '', oracion) for oracion in oraciones]  # No URLs
oraciones = [re.sub(r'[.,¿?¡!;:"()\[\]{}<>]', '', oracion.strip().lower()) for oracion in oraciones]  # Elimina puntuación 


In [29]:
# qgregamos <s> tres veces (modelo de tetegramas) y </s> 
from nltk.tokenize import TweetTokenizer
word_tokenizer = TweetTokenizer()

corpus = []
for sentence in oraciones:
    corpus += ["<s>"] + ["<s>"] + ["<s>"] + word_tokenizer.tokenize(sentence) + ["</s>"]

In [30]:
def tokenize_corpus(oraciones):
    word_tokenizer = TweetTokenizer()
    corpus = []
    for sentence in oraciones:
        corpus += ["<s>"] + ["<s>"] + ["<s>"] + word_tokenizer.tokenize(sentence) + ["</s>"]
    return corpus

Dividimos los datos en conjuto de entrenamiento, prueba y validación, ya que nos servira para el siguiente punto

In [31]:
import random 
from sklearn.model_selection import train_test_split

def split_data(oraciones):
    train_data, temp_data = train_test_split(oraciones, test_size=0.2, random_state=42)
    val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)
    return train_data, val_data, test_data

In [32]:
train_data, val_data, test_data = split_data(oraciones)
train = tokenize_corpus(train_data)
val = tokenize_corpus(val_data)
test = tokenize_corpus(test_data)

In [36]:
fdist = nltk.FreqDist(train)

In [37]:
def order_dic_frec(fdic):
    aux = [(fdic[key], key) for key in fdic]
    aux.sort()
    aux.reverse()
    return aux

In [38]:
V = order_dic_frec(fdist)
V = V[:10000]

In [40]:
dict_idx = {}
count = 0 
for _ , word in V:
    dict_idx[word] = count
    count += 1

In [162]:
# dict_idx

In [163]:
# Enmascaramso nuestro vocabulario
corpus_masked = ["<unk>" if word not in dict_idx else word for word in train]
# corpus_masked

In [43]:
#repetimos ya que no esta el token "<unk>" 
fdist2 = nltk.FreqDist(corpus_masked)
V = order_dic_frec(fdist2)

dict_idx = {}
count = 0 
for _ , word in V:
    dict_idx[word] = count
    count += 1

### 2. Modelos de lenguaje

Para el modelo de unigramas $P_{unigram}(w_n)$


In [45]:
P_unigram = {}
for count, word in V:
    P_unigram[word] = (count + 1) / (len(corpus_masked)+ len(V))
    

In [164]:
# P_unigram

In [47]:
P_unigram['<unk>']

0.06365963439383919

In [ ]:
# V

In [167]:
vocab_dict = {w: f for f, w in V}
# vocab_dict['política']

#### Unigramas

In [49]:
def prob_unigrams(word, corpus, vocab): 
    """
    Calcula la probabilidad de unigramas con suavizado de Laplace.
    Si la palabra no está en el vocabulario, se le asigna la probabilidad del token <unk>.
    """
    V = len(vocab)  
    N = len(corpus) 
    
    vocab_dict = {w: f for f, w in vocab}
    
    if word in vocab_dict:
        freq = vocab_dict[word]
        probability = (freq + 1) / (N + V)
    else:
        probability = 1 / (N + V)
    
    return probability

print(f"P('méxico') = {prob_unigrams('méxico', corpus_masked, V)}\n")
print(f"P('presidente') = {prob_unigrams('presidente', corpus_masked, V)}\n")
print(f"P('amlo') = {prob_unigrams('amlo', corpus_masked, V)}\n")
print(f"P('mejico') = {prob_unigrams('mejico', corpus_masked, V)}\n")  
print(f"P('peje') = {prob_unigrams('mejico', corpus_masked, V)}")  


P('méxico') = 0.002512449259623055

P('presidente') = 0.0023694442312480225

P('amlo') = 1.4307188439577802e-05

P('mejico') = 6.685602074569066e-08

P('peje') = 6.685602074569066e-08


#### Bigramas

In [50]:
bigrams = [(item[0],item[1]) for item in nltk.bigrams(corpus_masked)]
freqBigram = nltk.FreqDist(bigrams)

In [168]:
bigram = ['<s>'] + ['<s>']
# freqBigram[str(bigram)]

In [52]:
import math
from typing import Tuple

def prob_bigrams(bigrama: Tuple, freqBigram: dict, freqUnigram: dict, V: int, conditional: bool = False) -> float:
    """
    Calcula la probabilidad de un bigrama con suavizado de Laplace.
    Si el bigrama no está en el diccionario de frecuencias, se le asigna una probabilidad basada en el token <unk>.
    Si la primera palabra del bigrama no está en el diccionario de frecuencias de unigramas, se usa el token <unk>.
    """
    if conditional:
        if bigrama not in freqBigram:
            if bigrama[0] not in freqUnigram:
                prob = freqUnigram['<unk>']
            else:
                prob = 1 / (freqUnigram[bigrama[0]] + V)
        else:
            prob = (freqBigram[bigrama] + 1) / (freqUnigram[bigrama[0]] + V)

        return prob
    
    else: 
        if bigrama[1] in freqUnigram:
            log_prob = math.log(freqUnigram[bigrama[1]])
        else:
            log_prob = math.log(freqUnigram.get('<unk>', 1))
        
        if bigrama in freqBigram:
            log_prob += math.log((freqBigram[bigrama] + 1) / (freqUnigram[bigrama[0]] + V))
        else:
            log_prob += math.log(1 / (freqUnigram[bigrama[0]] + V))
        return math.exp(log_prob)

In [53]:
V_bigram = len(freqBigram)
bigram = bigrams[1]
print('P', bigram)
print(f"{prob_bigrams(bigram, freqBigram, P_unigram, V_bigram)}\n")

print('\nP(', bigram[1],'|',bigram[0],')')
print(f"{prob_bigrams(bigram, freqBigram, P_unigram, V_bigram, conditional=True)}\n")

P ('<s>', '<s>')
0.024137576646806082


P( <s> | <s> )
0.4902073301948379



In [54]:
bigram = bigrams[3]
print('P', bigram)
print(f"{prob_bigrams(bigram, freqBigram, P_unigram, V_bigram)}\n")

print('\nP(', bigram[1],'|',bigram[0],')')
print(f"{prob_bigrams(bigram, freqBigram, P_unigram, V_bigram, conditional=True)}\n")

P ('por', 'cierto')
1.613687886963897e-07


P( cierto | por )
0.0009424741268359506



In [55]:
bigram = ('<s>', '<peje>')
print('P', bigram)
print(f"{prob_bigrams(bigram, freqBigram, P_unigram, V_bigram)}\n")

print('\nP(', bigram[1],'|',bigram[0],')')
print(f"{prob_bigrams(bigram, freqBigram, P_unigram, V_bigram, conditional=True)}\n")

P ('<s>', '<peje>')
6.355673291374833e-08


P( <peje> | <s> )
9.983835678437272e-07



#### Trigramas

In [56]:
def prob_trigrams(trigrama: Tuple, freqTrigram: dict, freqBigram: dict, freqUnigram: dict, V: int, conditional: bool = False) -> float:
    """
    Calcula la probabilidad de un trigrama con suavizado de Laplace.
    Si el trigrama no está en el diccionario de frecuencias, se le asigna una probabilidad basada en el token <unk>.
    Si el bigrama (primeras dos palabras) no está en el diccionario de frecuencias de bigramas, se usa el token <unk>.
    Si alguna palabra no está en el diccionario de frecuencias de unigramas, se usa el token <unk>.
    """
    bigrama = (trigrama[0], trigrama[1])
    
    # Manejo de <unk> para la tercera palabra del trigrama
    freq_third_word = freqUnigram.get(trigrama[2], freqUnigram.get('<unk>', 1e-10))
    
    if conditional:
        if trigrama not in freqTrigram:
            if bigrama not in freqBigram:
                prob = 1 / (freqUnigram.get('<unk>', 1) + V)
            else:
                prob = 1 / (freqBigram[bigrama] + V)
        else:
            prob = (freqTrigram[trigrama] + 1) / (freqBigram[bigrama] + V)

        return prob
    
    else: 
        log_prob = np.log(freq_third_word)  
        log_prob += math.log(prob_bigrams(bigrama, freqBigram, freqUnigram, V, conditional=True)) 
        if trigrama in freqTrigram:
            log_prob += math.log((freqTrigram[trigrama] + 1) / (freqBigram[bigrama] + V))  
        else:
            log_prob += math.log(1 / (freqBigram[bigrama] + V))  

        return math.exp(log_prob)

In [169]:
# freqBigram.get(('<unk>', '<unk>'), 1)

In [58]:
trigrams = [(item[0],item[1],item[2]) for item in nltk.trigrams(corpus_masked)]
freqTrigram = nltk.FreqDist(trigrams)

In [170]:
# freqTrigram

In [60]:
# len(trigrams)

14947513

In [61]:
V_tigram = len(freqBigram)
trigram = trigrams[3]
print('P', trigram)
print(f"{prob_trigrams(trigram, freqTrigram, freqBigram, P_unigram, V_tigram)}\n")

print(f"P(' {trigram[2]},'|' {trigram[0]} , {trigram[1]} ')")
print(f"{prob_trigrams(trigram, freqTrigram, freqBigram,P_unigram,  V_tigram, conditional=True)}\n")


P ('por', 'cierto', 'méxico')
7.085601954341576e-12

P(' méxico,'|' por , cierto ')
2.992333641211217e-06



In [62]:
trigram = trigrams[1]
print('P', trigram)
print(f"{prob_trigrams(trigram, freqTrigram, freqBigram, P_unigram, V_tigram)}\n")

print(f"P(' {trigram[2]},'|' {trigram[0]} , {trigram[1]} ')")
print(f"{prob_trigrams(trigram, freqTrigram, freqBigram,P_unigram,  V_tigram, conditional=True)}\n")

P ('<s>', '<s>', 'por')
1.5480634938599396e-05

P(' por,'|' <s> , <s> ')
0.0039407243241577385



Si consideramos el trigrama con al menos una palabra fuera del voacbulario

In [63]:
trigram = ('<s>', '<peje>', '<amlo>')
print('P', trigram)
print(f"{prob_trigrams(trigram, freqTrigram, freqBigram, P_unigram, V_tigram)}\n")

print(f"P(' {trigram[2]},'|' {trigram[0]} , {trigram[1]} ')")
print(f"{prob_trigrams(trigram, freqTrigram, freqBigram,P_unigram,  V_tigram, conditional=True)}\n")

P ('<s>', '<peje>', '<amlo>')
6.34540008863135e-14

P(' <amlo>,'|' <s> , <peje> ')
9.983835534701996e-07



#### Tetragramas

In [65]:
def prob_tetragrams(tetragrama: Tuple, freqTetragram: dict, freqTrigram: dict, freqBigram: dict, freqUnigram: dict, V: int, conditional: bool = False) -> float:
    """
    Calcula la probabilidad de un tetragrama con suavizado de Laplace.
    Si el tetragrama no está en el diccionario de frecuencias, se le asigna una probabilidad basada en el token <unk>.
    Si el trigrama (primeras tres palabras) no está en el diccionario de frecuencias de trigramas, se usa el token <unk>.
    Si alguna palabra no está en el diccionario de frecuencias de unigramas, se usa el token <unk>.
    """
    trigrama = (tetragrama[0], tetragrama[1], tetragrama[2])
    
    # Manejo de <unk> para la cuarta palabra del tetragrama
    freq_fourth_word = freqUnigram.get(tetragrama[3], freqUnigram.get('<unk>', 1))
    
    if conditional:
        if tetragrama not in freqTetragram:
            # Si el tetragrama no está en el diccionario, usar suavizado de Laplace
            if trigrama not in freqTrigram:
                # Si el trigrama no está en el diccionario, usar <unk>
                prob = 1 / (freqUnigram.get('<unk>', 1) + V)
            else:
                prob = 1 / (freqTrigram[trigrama] + V)
        else:
            # Si el tetragrama está en el diccionario, calcular la probabilidad con suavizado de Laplace
            prob = (freqTetragram[tetragrama] + 1) / (freqTrigram[trigrama] + V)

        return prob
    
    else: 
        # Calcular la probabilidad en escala logarítmica
        log_prob = np.log(freq_fourth_word)  # Probabilidad del unigrama de la cuarta palabra
        log_prob += math.log(prob_trigrams(trigrama, freqTrigram, freqBigram, freqUnigram, V, conditional=True))  # Probabilidad del trigrama
        if tetragrama in freqTetragram:
            log_prob += math.log((freqTetragram[tetragrama] + 1) / (freqTrigram[trigrama] + V))  # Probabilidad condicional del tetragrama
        else:
            log_prob += math.log(1 / (freqTrigram[trigrama] + V))  # Suavizado de Laplace si el tetragrama no está en el diccionario
        
        return math.exp(log_prob)

In [66]:
from nltk import ngrams
tetragrams  = [(item[0],item[1],item[2],item[3]) for item in ngrams(corpus_masked,4)]
freqTetragrams  = nltk.FreqDist(tetragrams)

In [171]:
# freqTetragrams

In [68]:
V_tetra = len(freqTrigram)
tetragram = tetragrams[2]
print('P', tetragram)
print(f"{prob_tetragrams(tetragram, freqTetragrams, freqTrigram, freqBigram, P_unigram, V_tetra)}\n")

print(f"P(' {tetragram[3]},'|' {tetragram[0]} , {tetragram[1]}, {tetragram[2]}')")
print(f"{prob_tetragrams(tetragram, freqTetragrams, freqTrigram, freqBigram, P_unigram, V_tetra, conditional=True)}\n")


P ('<s>', 'por', 'cierto', 'méxico')
2.4481712340858714e-14

P(' méxico,'|' <s> , por, cierto')
6.56076283301612e-07



In [69]:
tetragram = ('andrés', 'manuel', 'lópez', 'obrador')
print('P', tetragram)
print(f"{prob_tetragrams(tetragram, freqTetragrams, freqTrigram, freqBigram, P_unigram, V_tetra)}\n")

print(f"P(' {tetragram[3]},'|' {tetragram[0]} , {tetragram[1]}, {tetragram[2]}')")
print(f"{prob_tetragrams(tetragram, freqTetragrams, freqTrigram, freqBigram, P_unigram, V_tetra, conditional=True)}\n")

P ('andrés', 'manuel', 'lópez', 'obrador')
9.882125638985205e-10

P(' obrador,'|' andrés , manuel, lópez')
0.0004950988896599584



In [70]:
tetragram = tetragrams[100]
print('P', tetragram)
print(f"{prob_tetragrams(tetragram, freqTetragrams, freqTrigram, freqBigram, P_unigram, V_tetra)}\n")

print(f"P(' {tetragram[3]},'|' {tetragram[0]} , {tetragram[1]}, {tetragram[2]}')")
print(f"{prob_tetragrams(tetragram, freqTetragrams, freqTrigram, freqBigram, P_unigram, V_tetra, conditional=True)}\n")

P ('<unk>', '</s>', '<s>', '<s>')
1.723674623807386e-06

P(' <s>,'|' <unk> , </s>, <s>')
0.005916579597398662



Ejemplo para palabra OOV 

In [71]:
tetragram = ('<s>', '<hola>', '<a>', '<amlo>')
print('P', tetragram)
print(f"{prob_tetragrams(tetragram, freqTetragrams, freqTrigram, freqBigram, P_unigram, V_tetra)}\n")

print(f"P(' {tetragram[3]},'|' {tetragram[0]} , {tetragram[1]}, {tetragram[2]}')")
print(f"{prob_tetragrams(tetragram, freqTetragrams, freqTrigram, freqBigram, P_unigram, V_tetra, conditional=True)}\n")

P ('<s>', '<hola>', '<a>', '<amlo>')
3.0446896350410135e-15

P(' <amlo>,'|' <s> , <hola>, <a>')
2.1869529579368843e-07



Dependiendo del modelo de n-gramas, podemos observar que la mayoría de las ocurrencias se concentran en unigramas y bigramas, mientras que los tetragramas son mucho menos frecuentes. De hecho, muchos tetragramas caen fuera del vocabulario, lo que reduce su presencia. Además, al aplicar un suavizamiento Laplaciano, redistribuimos la masa de probabilidad, lo que evita que cualquier n-grama tenga una probabilidad cercana a 1. A pesar de esto, la probabilidad sigue siendo relativamente alta, lo que indica que el modelo maneja bien la distribución de probabilidades incluso con el suavizamiento aplicado.

### 3. Construya un modelo interpolado

In [140]:
def probability_word(tetragrama, lambda1, lambda2, lambda3, lambda4): 
    term4 = lambda4 * P_unigram[tetragrama[3]] 
    term3 = lambda3 * prob_bigrams((tetragrama[2], tetragrama[3]), freqBigram, P_unigram, V_bigram, conditional=True)
    term2 = lambda2 * prob_trigrams((tetragrama[1], tetragrama[2], tetragrama[3]), freqTrigram, freqBigram,P_unigram,  V_tigram, conditional=True)
    term1 = lambda1 * prob_tetragrams(tetragrama, freqTetragrams, freqTrigram, freqBigram, P_unigram, V_tetra, conditional=True)
    return term1 + term2 + term3 + term4


def perplexity(lambda1, lambda2, lambda3, lambda4, corpus, s):
    N = len(corpus) #- 2*s
    log_prob_sum = 0
    
    for i in range(3, len(corpus)):
        tetragrama = (corpus[i-3], corpus[i-2], corpus[i-1], corpus[i])
        prob = probability_word(tetragrama, lambda1, lambda2, lambda3, lambda4)
        if prob > 0:  
            log_prob_sum += np.log2(prob) 
    
    H = (-1/N) * log_prob_sum
    
    # Calcular la perplejidad
    perplexity = 2 ** H
    return perplexity

In [82]:
validation = [ w if w in dict_idx else '<unk>' for w in val]
s = len(val_data)

In [193]:
lambda1 = 1/2;lambda2 = 1/2;lambda3 = 1/2;lambda4 = 1/2
perp = perplexity(lambda1, lambda2, lambda3, lambda4, validation,s )
print("lambda1={} lambda1={} lambda1={} lambda1={}, perp={}".format(lambda1, lambda2, lambda3, lambda4, perp))


lambda1=0.5 lambda1=0.5 lambda1=0.5 lambda1=0.5, perp=396.8949847238494


In [194]:
lambda1 = 0.1;lambda2 = 0.5;lambda3 = 0.4;lambda4 = 0.4
perp = perplexity(lambda1, lambda2, lambda3, lambda4, validation,s )
print("lambda1={} lambda1={} lambda1={} lambda1={}, perp={}".format(lambda1, lambda2, lambda3, lambda4, perp))

lambda1=0.1 lambda1=0.5 lambda1=0.4 lambda1=0.4, perp=492.02425057777526


In [195]:
lambda1 = 0.01;lambda2 = 0.5;lambda3 = 0.4;lambda4 = 0.4
perp = perplexity(lambda1, lambda2, lambda3, lambda4, validation,s )
print("lambda1={} lambda1={} lambda1={} lambda1={}, perp={}".format(lambda1, lambda2, lambda3, lambda4, perp))

lambda1=0.01 lambda1=0.5 lambda1=0.4 lambda1=0.4, perp=492.5497847746865


In [196]:
lambda1 = 0.8;lambda2 = 0.1;lambda3 = 0.1;lambda4 = 0.1
perp = perplexity(lambda1, lambda2, lambda3, lambda4, validation,s )
print("lambda1={} lambda1={} lambda1={} lambda1={}, perp={}".format(lambda1, lambda2, lambda3, lambda4, perp))

lambda1=0.8 lambda1=0.1 lambda1=0.1 lambda1=0.1, perp=1927.4456207989977


In [197]:
lambda1 = 0.01;lambda2 = 0.5;lambda3 = 0.6;lambda4 = 0.7
perp = perplexity(lambda1, lambda2, lambda3, lambda4, validation,s )
print("lambda1={} lambda1={} lambda1={} lambda1={}, perp={}".format(lambda1, lambda2, lambda3, lambda4, perp))

lambda1=0.01 lambda1=0.5 lambda1=0.6 lambda1=0.7, perp=304.66508583253966


In [198]:
lambda1 = 0.01;lambda2 = 0.02;lambda3 = 0.8;lambda4 = 0.9
perp = perplexity(lambda1, lambda2, lambda3, lambda4, validation,s )
print("lambda1={} lambda1={} lambda1={} lambda1={}, perp={}".format(lambda1, lambda2, lambda3, lambda4, perp))

lambda1=0.01 lambda1=0.02 lambda1=0.8 lambda1=0.9, perp=244.47221273188995


In [199]:
lambda1 = 0.01;lambda2 = 0.1;lambda3 = 0.8;lambda4 = 0.9
perp = perplexity(lambda1, lambda2, lambda3, lambda4, validation,s )
print("lambda1={} lambda1={} lambda1={} lambda1={}, perp={}".format(lambda1, lambda2, lambda3, lambda4, perp))

lambda1=0.01 lambda1=0.1 lambda1=0.8 lambda1=0.9, perp=242.97334217338215


Para el conjunto de test, vamos a considerar $ \vec{\lambda} = [0.01,0.1,0.8,0.9]$ ya que generó valor bajo de perplejidad

In [200]:
test = [ w if w in dict_idx else '<unk>' for w in test]
s2 = len(test_data)

In [201]:
lambda1 = 0.01;lambda2 = 0.1;lambda3 = 0.8;lambda4 = 0.9
perp = perplexity(lambda1, lambda2, lambda3, lambda4, test, s2)
print("lambda1={} lambda1={} lambda1={} lambda1={}. perp={}".format(lambda1, lambda2, lambda3, lambda4, perp))

lambda1=0.01 lambda1=0.1 lambda1=0.8 lambda1=0.9. perp=243.64053902720764


Podemos observar que sobre el conjunto de test también se obtiene una perplejidad muy buena, ya que es bajo a comparación de las demás, por lo que dichos lambda son una buena elección para nuestro modelo. 

### Generación de Texto

#### 1. Algoritmo EM

Para este punto nquiero citar la canción de Espinoza Paz "Lo Intentamos" por su frase "Lo intentamos pero no pudo funcionar" :(


#### 2. Función generar texto

Consideramos los valores fijos de $ \vec{\lambda} = [0.01,0.1,0.8,0.9]$ ya que generó valor bajo de perplejidad. 

In [149]:
next(iter(freqTetragrams))

('</s>', '<s>', '<s>', '<s>')

In [156]:
# def generar_next_toke(tetragrama, evitar_token, freqTetragrams):
#     """
#     Genera el siguiente token basándose en el modelo de lenguaje.
    
#     Args: 
#         tetragrama: tuple de 4 tokens, por ejemplo ('</s>', '<s>', '<s>', '<s>')
#         evitar_token: token que se debe evitar (puede ser None)
#         freqTetragrams: diccionario de frecuencias de los tetragramas
#     Returns:
#         El token candidato con mayor probabilidad
#     """
#     mejor_token = None
#     mejor_prob = 0.0
    
#     # Se extraen los tres últimos tokens del tetragrama actual para definir el contexto
#     contexto = (tetragrama[1], tetragrama[2], tetragrama[3])
    
#     # Se itera sobre cada tetragrama disponible en el diccionario de frecuencias
#     for key in freqTetragrams.keys():
#         # Se verifica que los tres primeros tokens de la clave coincidan con el contexto
#         if key[0:3] == contexto:
#             candidato = key[3]
#             # Si se define evitar_token y el candidato coincide, se omite
#             if evitar_token is not None and candidato == evitar_token:
#                 continue
#             # Se calcula la probabilidad del tetragrama formado por el contexto y el candidato
#             prob = probability_word((contexto[0], contexto[1], contexto[2], candidato), lambda1, lambda2, lambda3, lambda4)
#             # Se actualiza el token candidato si se encuentra una mayor probabilidad
#             if prob > mejor_prob:
#                 mejor_prob = prob
#                 mejor_token = candidato
#     return mejor_token


# def generar_text():
#     """
#     Genera texto utilizando la función generar_next_toke para obtener cada token.
    
#     El proceso se detiene cuando se genera el token de terminación '</s>' o se han generado 50 tokens.
#     """
#     # Inicializamos el tetragrama de inicio (usando '<s>' como símbolo de inicio)
#     tetragrama_actual = ('<s>', '<s>', '<s>', '<s>')
#     tokens_generados = []
    
#     # Se generan tokens hasta alcanzar 50 o hasta generar el token de terminación
#     while len(tokens_generados) < 50:
#         siguiente_token = generar_next_toke(tetragrama_actual, None, freqTetragrams)
#         if not siguiente_token:
#             break  # Si no se puede generar un token, se termina el proceso
#         tokens_generados.append(siguiente_token)
#         # Si se genera el token de terminación, se interrumpe el ciclo
#         if siguiente_token == "</s>":
#             break
#         # Se actualiza el tetragrama actual desplazándolo a la izquierda e incorporando el nuevo token
#         tetragrama_actual = (tetragrama_actual[1], tetragrama_actual[2], tetragrama_actual[3], siguiente_token)
    
#     # Se retorna el texto generado (los tokens se unen separados por espacios)
#     return " ".join(tokens_generados)

## El Ahorcado

In [178]:
import wget 

# Texto paara estimar P(word)
url_pag = 'http://norvig.com/big.txt'
wget.download(url = url_pag)

100% [..........................................................................] 6488666 / 6488666

'big (2).txt'

In [180]:
# Función para extraer palabras
def words(text):
    return re.findall('[^\d\W]+', text.lower())

# Cargar el corpus y contar las palabras
WORDS = Counter(words(open('big (2).txt').read()))

# Función de probabilidad
def P(word, N=sum(WORDS.values())):
    return WORDS[word] / N

def hangman(word):
    return max(candidates(word), key=P)

def candidates(word):
    return (known([word]) or 
            known(ahorcado_edit1(word)) or 
            known(ahorcado_edit2(word)) or 
            known(ahorcado_edit3(word)) or 
            known(ahorcado_edit4(word)) or 
            [word])

# Función para filtrar palabras conocidas
def known(words):
    return set(w for w in words if w in WORDS)

# Funciones para generar candidatos con caracteres faltantes
def ahorcado_edit1(word):
    letters = 'abcdefghijklmnopqrstuvwxyz'
    splits = [(word[:i], word[i:]) for i in range(len(word)) if word[i] == '_']
    replaces = [L + c + R[1:] for L, R in splits if R for c in letters]
    return set(replaces)

def ahorcado_edit2(word):
    return (e2 for e1 in ahorcado_edit1(word) for e2 in ahorcado_edit1(e1))

def ahorcado_edit3(word):
    return (e2 for e1 in ahorcado_edit2(word) for e2 in ahorcado_edit1(e1))

def ahorcado_edit4(word):
    return (e2 for e1 in ahorcado_edit3(word) for e2 in ahorcado_edit1(e1))

<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ericl\AppData\Local\Temp\ipykernel_15376\4207279055.py:3: SyntaxWarning: invalid escape sequence '\d'
  return re.findall('[^\d\W]+', text.lower())


In [183]:
%%time
word = hangman ( "pe_p_e" )
word

CPU times: total: 0 ns
Wall time: 0 ns


'people'

In [181]:
%%time
word = hangman('sp_l_i__')
word

CPU times: total: 2.67 s
Wall time: 2.86 s


'spelling'

In [182]:
%%time 
word = hangman("si_nif_c_nc_")
word

CPU times: total: 2.95 s
Wall time: 3.07 s


'significance'

In [184]:
%%time
word = hangman("l_br_ry")
word

CPU times: total: 0 ns
Wall time: 0 ns


'library'

In [185]:
%%time
word = hangman("_x_mpl_")
word

CPU times: total: 0 ns
Wall time: 35.4 ms


'example'

In [186]:
%%time
word = hangman("h_ll_")
word

CPU times: total: 0 ns
Wall time: 6.01 ms


'hills'

In [187]:
%%time
word = hangman("hell_")
word

CPU times: total: 0 ns
Wall time: 1.5 ms


'hello'

In [188]:
%%time
word = hangman("b_n_n_")
word

CPU times: total: 15.6 ms
Wall time: 30.3 ms


'banana'

In [189]:
%%time
word = hangman("a__l_")
word

CPU times: total: 0 ns
Wall time: 30.3 ms


'apply'

In [191]:
%%time
word = hangman("p_zz_")
word

CPU times: total: 0 ns
Wall time: 0 ns


'p_zz_'

La estrategia es generar todas las posibles combinaciones de palabras que podrían encajar en el patrón dado (con hasta 4 caracteres faltantes) y luego seleccionar la palabra más probable según el modelo de lenguaje basado en la frecuencia de palabras. Aunque esta estrategia tiene el inconveniente de que si alguna de las palabras no está en el corpus, la función podría devolver una palabra incorrecta o la misma entrada como en el caso de pizza.

#### Comentario

Combinando el enfoque descrito Norvig con un LLM podemos mejorar la prediccion (reducir los errores gramaticales) de las palabras. Norvig sugiere correcciones basadas en similitud ortográfica y frecuencia de palabras, pero no considera el contexto, solo se consideran las frecuecnias que aparece la palabra en el texto. El LLM puedo ayudar a mejorar la coherencia gramatical y semántica de la frase completa.

Por ejemplo, para corregir "off", generamos candidatos como "of", "on" y "or". Luego, el LM calcula cuál opción tiene mas sentido en la frase. En este caso, "of" seria la corrección más probable porque maximza la coherencia de la frase.
La ventaja es que no solo corregimos ortografía, sino también errores contextuales, como confusiones entre palabras similares (por ejemplo, "their" vs "there"). Podriamos utilixar un modelo preentrenados como GPT-2 con un simple pipeline. 